In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False

df = pd.read_csv("7.csv ")
# 划分特征和目标变量
X = df.drop(['gb'], axis=1)
y = df['gb']
# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42,stratify=df['gb'])
df.head()

,ID,DPN,DR,BMI,gender,Age,Smoke,drink,Duration.of.diabetes,Family.history.of.T2DM,...,D25,FT3,FT4,TSH,DN,FLD,Vfa,sfa,METS-IR,gb
0,182,0,0,27.30,1,63,0,0,3,0,...,22.45,6.04,15.56,3.67,1,0,119,231,42.65,1
1,212,0,0,25.00,1,63,0,0,3,0,...,6.12,5.73,13.62,2.43,0,0,92,189,40.35,1
2,438,1,0,20.70,0,80,0,0,3,0,...,19.40,4.85,21.79,3.14,1,0,108,175,31.96,1
3,472,1,1,23.05,1,74,0,0,3,0,...,16.30,4.32,12.49,1.13,1,0,80,156,38.15,1
4,474,1,1,29.40,0,74,0,0,3,0,...,19.80,5.93,15.97,9.96,0,1,145,234,47.83,1


In [16]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold
# 初始化随机森林分类器
clf = RandomForestClassifier(random_state=42)

# 定义StratifiedKFold用于交叉验证
cv = StratifiedKFold(n_splits=5)

# 递归特征消除和交叉验证
rfecv = RFECV(estimator=clf, step=1, cv=cv, scoring='accuracy')
rfecv.fit(X_train, y_train)

# 打印最佳特征数量
print(f"Optimal number of features: {rfecv.n_features_}")

# 获取交叉验证每一折的分数
cv_results = rfecv.cv_results_

# 取出5次交叉验证的单独分数
fold_scores = [cv_results[f'split{i}_test_score'] for i in range(5)]
mean_scores = cv_results['mean_test_score']  # 计算平均得分
# 输出选择的特征列
selected_features = X_train.columns[rfecv.support_]
print(f"Selected features: {list(selected_features)}")
df_selected = df[selected_features]
df_selected.head()

Optimal number of features: 12
Selected features: ['Age', 'WBC', 'ALB', 'BUN', 'eGFR', 'MALB', 'ACR', 'HbAlc', 'PTH', 'FT4', 'TSH', 'METS-IR']


,Age,WBC,ALB,BUN,eGFR,MALB,ACR,HbAlc,PTH,FT4,TSH,METS-IR
0,63,5.2,43.1,7.13,61.44,3.20,75.48,6.4,64.2,15.56,3.67,42.65
1,63,6.9,40.8,3.73,110.20,0.86,10.63,5.8,118.9,13.62,2.43,40.35
2,80,4.8,40.0,6.13,104.89,32.14,651.92,10.3,21.4,21.79,3.14,31.96
3,74,6.6,31.2,8.51,45.72,259.02,5928.27,11.3,67.1,12.49,1.13,38.15
4,74,9.7,42.3,3.86,112.39,1.09,23.55,10.6,23.6,15.97,9.96,47.83


In [5]:
plt.figure(figsize=(12, 8), dpi=1200)
plt.title('Recursive Feature Elimination with Cross-Validation (RFCV)', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Number of features selected', fontsize=14, labelpad=15)
plt.ylabel('Cross-validation score (accuracy)', fontsize=14, labelpad=15)
# 设置背景颜色
plt.gca().set_facecolor('#f7f7f7')
# 绘制每一条灰色线，表示5次交叉验证
for i in range(5):
    plt.plot(range(1, len(fold_scores[i]) + 1), fold_scores[i], marker='o', color='gray', linestyle='-', 
             linewidth=0.8, alpha=0.6)
# 绘制淡黑色线，表示平均交叉验证得分
plt.plot(range(1, len(mean_scores) + 1), mean_scores, marker='o', color='#696969', linestyle='-', 
         linewidth=3, label='Mean CV Accuracy')
# 绘制最佳特征数的垂直线
plt.axvline(x=rfecv.n_features_, color='#E76F51', linestyle='--', linewidth=2, label=f'Optimal = {rfecv.n_features_}')
plt.legend(fontsize=12, loc='best', frameon=True, shadow=True, facecolor='white', framealpha=0.9)
plt.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.subplots_adjust(left=0.1, right=0.9, top=0.9, bottom=0.1)
plt.savefig('分类.pdf', format='pdf', bbox_inches='tight')
plt.show()